In [1]:
# Read directory.msgpack

import msgpack

directory_path = "../data/azoflip_data/MSGpack/directory.msgpack"
with open(directory_path, "rb") as directory_file:
    dic = next(msgpack.Unpacker(directory_file, strict_map_key=False))


In [2]:
# Find all the msgpacks that contain unsubstituted azobenzene

azobenzene_inchikeys = {
    "Z": "DMLAVOWQYNRWNQ-YPKPFQOONA-N",
    "E": "DMLAVOWQYNRWNQ-BUHFOSPRNA-N",
}
target_inchikeys = set(azobenzene_inchikeys.values())

azobenzene_msgpacks = {}
isomer_counts = {isomer: 0 for isomer in azobenzene_inchikeys}

no_azobenzene_msgpacks = []
for msgpack_name, geometries in dic.items():
    geometry_ids = []
    for geometry_id, geometry in geometries.items():
        inchikey = geometry["species"]["inchikey"]
        if inchikey in target_inchikeys:
            geometry_ids.append(geometry_id)
            isomer = next(
                label for label, key in azobenzene_inchikeys.items() if key == inchikey
            )
            isomer_counts[isomer] += 1
    if geometry_ids:
        azobenzene_msgpacks[msgpack_name] = geometry_ids
    else:
        no_azobenzene_msgpacks.append(msgpack_name)

assert all(
    dic[msgpack_name][geometry_id]["species"]["inchikey"] in target_inchikeys
    for msgpack_name, geometry_ids in azobenzene_msgpacks.items()
    for geometry_id in geometry_ids
)

total_geometries = sum(map(len, azobenzene_msgpacks.values()))
assert total_geometries == 91_321
assert isomer_counts == {"Z": 20_376, "E": 70_945}

print(f"MessagePack files containing azobenzene geometries: {len(azobenzene_msgpacks)}")
print(f"Total geometries: {total_geometries:,}")
print(f"Z: {isomer_counts['Z']:,}; E: {isomer_counts['E']:,}")
for msgpack_name, geometry_ids in sorted(azobenzene_msgpacks.items()):
    print(f"{msgpack_name}: {len(geometry_ids):,}")
print(f"MessagePack files without azobenzene geometries: {len(no_azobenzene_msgpacks)}")


MessagePack files containing azobenzene geometries: 962
Total geometries: 91,321
Z: 20,376; E: 70,945
0000.msgpack: 657
0001.msgpack: 500
0002.msgpack: 501
0003.msgpack: 500
0004.msgpack: 500
0005.msgpack: 501
0006.msgpack: 501
0007.msgpack: 500
0008.msgpack: 500
0009.msgpack: 501
0010.msgpack: 500
0011.msgpack: 266
0012.msgpack: 426
0013.msgpack: 486
0014.msgpack: 499
0016.msgpack: 8
0017.msgpack: 496
0018.msgpack: 500
0019.msgpack: 501
0020.msgpack: 159
0021.msgpack: 243
0022.msgpack: 50
0023.msgpack: 449
0024.msgpack: 221
0025.msgpack: 68
0026.msgpack: 303
0027.msgpack: 297
0028.msgpack: 12
0029.msgpack: 296
0030.msgpack: 253
0031.msgpack: 191
0032.msgpack: 148
0033.msgpack: 281
0034.msgpack: 119
0035.msgpack: 223
0036.msgpack: 158
0037.msgpack: 169
0038.msgpack: 350
0048.msgpack: 1
0049.msgpack: 2
0050.msgpack: 2
0051.msgpack: 8
0052.msgpack: 26
0053.msgpack: 33
0054.msgpack: 3
0055.msgpack: 41
0056.msgpack: 2
0057.msgpack: 3
0058.msgpack: 35
0059.msgpack: 5
0060.msgpack: 7
0061.ms

In [3]:
# Read one msgpack sub-directory

import msgpack

path = '../data/azoflip_data/MSGpack/1367.msgpack'
with open(path, "rb") as msgpack_file:
    directory = next(msgpack.Unpacker(msgpack_file, strict_map_key=False))

print(f"This sub-dictionary has length {len(directory)}")


KeyboardInterrupt: 

In [4]:
# Write XYZ file with energies and forces

import json

atomic_symbols = {1: "H", 6: "C", 7: "N"}
output_path = "../data/azoflip_data/XYZ/1367_ground_first_excited.xyz"
skipped_geometry_ids = []

with open(output_path, "w") as xyz_file:
    for geometry_id, geometry in directory.items():
        coordinates = geometry["xyz"]
        ground_energy = geometry["props"].get("totalenergy")
        ground_forces = geometry["props"].get("forces")
        excited_states = geometry["props"].get("excitedstates", [])
        first_excited_state = excited_states[0] if excited_states else {}
        first_excited_energy = first_excited_state.get("energy")
        first_excited_forces = first_excited_state.get("forces")

        energies = [[ground_energy, first_excited_energy]]
        forces = [ground_forces, first_excited_forces]

        if (
            any(energy is None for energy in energies[0])
            or any(force is None or len(force) != len(coordinates) for force in forces)
        ):
            skipped_geometry_ids.append(geometry_id)
            continue

        # X-MACE expects forces as [atom, state, Cartesian component].
        forces_by_atom = [
            [ground_force, excited_force]
            for ground_force, excited_force in zip(ground_forces, first_excited_forces)
        ]

        xyz_file.write(f"{len(coordinates)}\n")
        xyz_file.write(
            "Properties=species:S:1:pos:R:3 "
            f'REF_energy="_JSON {json.dumps(energies)}" '
            f'REF_forces="_JSON {json.dumps(forces_by_atom)}"\n'
        )
        for atomic_number, x, y, z in coordinates:
            symbol = atomic_symbols[int(atomic_number)]
            xyz_file.write(f"{symbol:<2} {x: .8f} {y: .8f} {z: .8f}\n")

written_geometries = len(directory) - len(skipped_geometry_ids)
print(f"Wrote {written_geometries} geometries to {output_path}")
print(f"Skipped {len(skipped_geometry_ids)} geometries without complete energy/force data")
if skipped_geometry_ids:
    print(f"Skipped geometry IDs: {skipped_geometry_ids}")


Wrote 461 geometries to ../data/azoflip_data/XYZ/1367_ground_first_excited.xyz
Skipped 76 geometries without complete energy/force data
Skipped geometry IDs: [83069875, 56321748, 56354967, 56760963, 56796844, 56785808, 56838834, 58512212, 57000185, 83674453, 81662905, 81662897, 83054009, 55202629, 55295245, 53588863, 53588865, 53588866, 53588867, 53590516, 53590517, 58292036, 58292037, 58292038, 58292039, 53590518, 53590519, 53590523, 53590525, 58292025, 58292026, 58292027, 58292032, 58292033, 58292034, 55263815, 56354962, 56761449, 56999939, 57114828, 56992010, 84490877, 56391599, 56404985, 56794948, 84490959, 56809207, 55433831, 55449126, 55509833, 55510444, 55775496, 56069544, 58292029, 56373711, 56193092, 56354619, 56322376, 56330491, 56998201, 54147706, 58292024, 56760873, 53588868, 53590521, 55401235, 56321755, 53588864, 53590520, 58292028, 58292030, 58292031, 58292035, 53590522, 53590524, 53590526]


In [4]:
# Repair existing XYZ files that store REF_forces as [state, atom, Cartesian].
# This cell creates a one-time backup, then rewrites each file as
# [atom, state, Cartesian], which X-MACE expects.

from pathlib import Path
import shutil

import ase.io
import numpy as np

xyz_paths = [
    Path("../data/azoflip_data/XYZ/cis_trans_azobenzenes.xyz"),
    Path("../data/azoflip_data/XYZ/1367_ground_first_excited.xyz"),
]

for xyz_path in xyz_paths:
    frames = ase.io.read(xyz_path, index=":")
    corrected_frames = 0
    for frame_index, atoms in enumerate(frames):
        forces = np.asarray(atoms.info["REF_forces"])
        if forces.ndim != 3 or forces.shape[2] != 3:
            raise ValueError(
                f"{xyz_path}, frame {frame_index}: REF_forces must be three-dimensional, "
                f"with three Cartesian components; got {forces.shape}"
            )
        if forces.shape[0] == len(atoms):
            continue  # Already [atom, state, Cartesian].
        if forces.shape[1] != len(atoms):
            raise ValueError(
                f"{xyz_path}, frame {frame_index}: cannot identify the atom axis in "
                f"REF_forces with shape {forces.shape}"
            )
        atoms.info["REF_forces"] = np.transpose(forces, (1, 0, 2))
        corrected_frames += 1

    if corrected_frames:
        backup_path = xyz_path.with_name(f"{xyz_path.stem}_state_first_backup{xyz_path.suffix}")
        if not backup_path.exists():
            shutil.copy2(xyz_path, backup_path)
        ase.io.write(xyz_path, frames, format="extxyz")
        print(f"Rewrote {corrected_frames} frames in {xyz_path}; backup: {backup_path}")
    else:
        print(f"{xyz_path} is already atom-first; no changes made.")


Rewrote 17260 frames in ../data/azoflip_data/XYZ/cis_trans_azobenzenes.xyz; backup: ../data/azoflip_data/XYZ/cis_trans_azobenzenes_state_first_backup.xyz
Rewrote 461 frames in ../data/azoflip_data/XYZ/1367_ground_first_excited.xyz; backup: ../data/azoflip_data/XYZ/1367_ground_first_excited_state_first_backup.xyz
